# Day 05 下午学生项目：电商用户多维分析

**小组编号：** 请填写  
**成员：** 请填写  
**专题方向：** A / B / C / D / E

> 请只在标有 `TODO` 的区域填写代码，不要删除任务说明、检查点和反思题。

## 实验目标与提交要求

你需要完成：

1. 数据加载与验收；
2. 公共基础指标；
3. 一个单维专题分析；
4. 一个双维交叉分析；
5. 三个CSV报表；
6. 至少3条结论、1条限制和1项建议。

**重要边界：** 一行是一名用户；返现不是消费金额；相关不等于因果。

## 任务0：小组配置

In [6]:
from pathlib import Path
import pandas as pd
import numpy as np
try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

GROUP_ID = "505/18"
MEMBERS = ["11"]
TOPIC = "A"

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

def find_workspace_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "output" / "day04_project" / "ecommerce_customer_cleaned.csv").exists():
            return candidate
    raise FileNotFoundError("未找到清洗后数据，请检查项目目录。")

ROOT = find_workspace_root()
DATA_PATH = ROOT / "output" / "day04_project" / "ecommerce_customer_cleaned.csv"
OUTPUT_DIR = ROOT / "output" / "day05_student" / GROUP_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("小组：", GROUP_ID, MEMBERS)
print("专题：", TOPIC)
print("输入：", DATA_PATH)
print("输出：", OUTPUT_DIR)

小组： 505/18 ['钟元']
专题： A
输入： C:\Users\DELL\PycharmProjects\python\output\day04_project\ecommerce_customer_cleaned.csv
输出： C:\Users\DELL\PycharmProjects\python\output\day05_student\505\18


### 检查点0

- [✅️] 已填写组号、成员和专题；
- [✅️] Notebook文件名包含组号；
- [✅️] 输出目录中的组号正确。

## 任务1：加载并验收数据（必做）

In [8]:
# TODO 1：读取清洗后的CSV，变量名必须为df
df = pd.read_csv(DATA_PATH)
print("数据读取完成，df类型：", type(df))

# TODO 2：输出shape、前5行和字段类型
print("数据集shape（行数，列数）：", df.shape)
print("\n前5行数据：")
display(df.head())
print("\n各个字段的数据类型：")
print(df.dtypes)

# TODO 3：计算验收结果
# 定义核心字段，和你最开始设置的core_cols保持一致
core_cols = [
    "CustomerID", "Churn", "Tenure", "TenureGroup", "OrderCount",
    "CouponUsed", "CashbackAmount", "DaySinceLastOrder", "Complain",
    "PreferedOrderCat", "PreferredPaymentMode"
]

validation = {
    "行数": len(df),
    "列数": df.shape[1],
    "CustomerID重复数": int(df["CustomerID"].duplicated().sum()),
    "核心字段缺失数": int(df[core_cols].isna().sum().sum()),
    "Churn取值": sorted(df["Churn"].unique().tolist())
}
validation

数据读取完成，df类型： <class 'pandas.DataFrame'>
数据集shape（行数，列数）： (5630, 22)

前5行数据：


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount,TenureGroup,IsMobileLogin
0,50001,1,4.00,Mobile Phone,3,6.00,Debit Card,Female,3.00,3,Laptop & Accessory,2,Single,9,1,11.00,1.00,1.00,5.00,159.93,0‑12个月,1
1,50002,1,9.00,Mobile Phone,1,8.00,UPI,Male,3.00,4,Mobile Phone,3,Single,7,1,15.00,0.00,1.00,0.00,120.90,0‑12个月,1
2,50003,1,9.00,Mobile Phone,1,30.00,Debit Card,Male,2.00,4,Mobile Phone,3,Single,6,1,14.00,0.00,1.00,3.00,120.28,0‑12个月,1
3,50004,1,0.00,Mobile Phone,3,15.00,Debit Card,Male,2.00,4,Laptop & Accessory,5,Single,8,0,23.00,0.00,1.00,3.00,134.07,0‑12个月,1
4,50005,1,0.00,Mobile Phone,1,12.00,Credit Card,Male,3.00,3,Mobile Phone,5,Single,3,0,11.00,1.00,1.00,3.00,129.60,0‑12个月,1



各个字段的数据类型：
CustomerID                       int64
Churn                            int64
Tenure                         float64
PreferredLoginDevice               str
CityTier                         int64
WarehouseToHome                float64
PreferredPaymentMode               str
Gender                             str
HourSpendOnApp                 float64
NumberOfDeviceRegistered         int64
PreferedOrderCat                   str
SatisfactionScore                int64
MaritalStatus                      str
NumberOfAddress                  int64
Complain                         int64
OrderAmountHikeFromlastYear    float64
CouponUsed                     float64
OrderCount                     float64
DaySinceLastOrder              float64
CashbackAmount                 float64
TenureGroup                        str
IsMobileLogin                    int64
dtype: object


{'行数': 5630, '列数': 22, 'CustomerID重复数': 0, '核心字段缺失数': 0, 'Churn取值': [0, 1]}

In [9]:
# 完成上一个单元后再运行本检查点
assert isinstance(df, pd.DataFrame), "df还不是DataFrame"
assert df.shape == (5630, 22), "数据形状应为(5630, 22)"
assert df["CustomerID"].is_unique, "CustomerID应唯一"
assert set(df["Churn"].unique()) == {0, 1}, "Churn应只包含0和1"
print("检查点1通过")

检查点1通过


**数据粒度：** 请用一句话填写：数据为单用户粒度，一行对应唯一一位客户，记录每位客户的基础属性、APP使用习惯、订单消费情况以及流失标签。
____________________________________________________________

## 任务2：公共基础指标（必做）

In [25]:
# TODO：构建overall_metrics DataFrame，至少包含以下指标：
# 计算各个指标
total_user = len(df)
churn_count = df["Churn"].sum()
churn_rate = df["Churn"].mean()
avg_order = df["OrderCount"].mean()
median_order = df["OrderCount"].median()
avg_coupon = df["CouponUsed"].mean()
avg_cashback = df["CashbackAmount"].mean()
avg_app_time = df["HourSpendOnApp"].mean()
avg_satis = df["SatisfactionScore"].mean()
avg_lastday = df["DaySinceLastOrder"].mean()

# 构建10行2列的DataFrame，len(overall_metrics)=10，满足断言条件
overall_metrics = pd.DataFrame({
    "指标名称": [
        "用户数",
        "流失人数",
        "流失率",
        "平均订单数",
        "订单数中位数",
        "平均优惠券数",
        "平均返现",
        "平均App时长",
        "平均满意度",
        "平均距上次下单天数"
    ],
    "指标值": [
        total_user,
        churn_count,
        churn_rate,
        avg_order,
        median_order,
        avg_coupon,
        avg_cashback,
        avg_app_time,
        avg_satis,
        avg_lastday
    ]
})
display(overall_metrics)

# 赋值总体流失率
overall_churn_rate = df["Churn"].mean()


,指标名称,指标值
0,用户数,"5,630.00"
1,流失人数,948.00
2,流失率,0.17
3,平均订单数,2.96
4,订单数中位数,2.00
5,平均优惠券数,1.72
6,平均返现,177.22
7,平均App时长,2.93
8,平均满意度,3.07
9,平均距上次下单天数,4.46


In [26]:
# 检查点2
assert isinstance(overall_metrics, pd.DataFrame), "overall_metrics应为DataFrame"
assert len(overall_metrics) >= 10, "公共指标至少10项"

# TODO：将下面变量赋值为你计算的总体流失率
assert abs(overall_churn_rate - 0.16838365896980462) < 1e-8, "总体流失率不正确"
print("检查点2通过")

检查点2通过


## 任务3：单维专题分析（必做）

请选择一个专题：

- A：`TenureGroup` 用户生命周期；
- B：`Complain` 或 `SatisfactionScore` 服务体验；
- C：`PreferedOrderCat` 品类与订单；
- D：`PreferredPaymentMode` 支付与优惠；
- E：`CityTier` 或 `PreferredLoginDevice` 城市与设备。

最低要求：使用 `groupby + agg`，同时输出用户数和至少3项业务指标。

In [27]:
# TODO：填写你的分组字段
segment_field = "TenureGroup"

# TODO：使用groupby + agg完成命名聚合
segment_analysis = df.groupby(segment_field).agg(
    用户数 = ("CustomerID", "count"),
    流失人数 = ("Churn", "sum"),
    流失率 = ("Churn", "mean"),
    平均订单数 = ("OrderCount", "mean"),
    平均返现 = ("CashbackAmount", "mean")
).reset_index()

# TODO：重置索引、排序并展示
segment_analysis = segment_analysis.sort_values("流失率", ascending=False)
display(segment_analysis)

,TenureGroup,用户数,流失人数,流失率,平均订单数,平均返现
0,0‑12个月,3552,846,0.24,2.56,159.99
1,12‑24个月,1574,102,0.06,3.64,200.72
2,24‑36个月,500,0,0.00,3.70,225.29
3,36个月以上,4,0,0.00,2.00,226.38


In [28]:
# 检查点3
assert segment_field in df.columns, "segment_field不是有效字段"
assert isinstance(segment_analysis, pd.DataFrame), "segment_analysis应为DataFrame"
assert "用户数" in segment_analysis.columns, "专题表必须包含用户数"
assert len(segment_analysis) >= 2, "专题分析至少应有两个分组"
print("检查点3通过")

检查点3通过


### 专题分析记录

**数据现象：**
随着用户在平台的使用时长增加，客户流失率明显下降：0‑12 个月的新用户群体流失率高达 24%；12‑24 个月用户流失率下降至 6%；24‑36 个月以及 36 个月以上老用户流失率为 0；同时长期用户的平均订单数量、平均返现金额整体高于新用户群体。

**可能解释：**  
用户使用时长和流失风险呈现负相关，新用户更容易流失可能和新人阶段下单体验较差、优惠券福利偏少有关；长期用户订单频次更高、获得返现更多，和用户留存情况向好相关；该推论还需要结合用户投诉情况、满意度得分进一步验证。

## 任务4：双维度交叉分析（必做）

In [29]:
# TODO：从以下维度中选择两个
# TenureGroup、Complain、PreferedOrderCat、CityTier、PreferredLoginDevice
dim_1 = "TenureGroup"
dim_2 = "Complain"

# TODO：双维度分组聚合：用户数、流失人数、流失率，再加1个行为指标平均订单数OrderCount
cross_analysis = df.groupby([dim_1, dim_2]).agg(
    用户数 = ("CustomerID", "count"),
    流失人数 = ("Churn", "sum"),
    流失率 = ("Churn", "mean"),
    平均订单数 = ("OrderCount", "mean")
).reset_index()

# TODO：新增样本提示列：用户数<30标记为"小样本"，否则"可观察"
cross_analysis["样本提示"] = cross_analysis["用户数"].apply(lambda x: "小样本" if x < 30 else "可观察")

# TODO：按照流失率降序排序
cross_analysis = cross_analysis.sort_values("流失率", ascending=False)
display(cross_analysis)

,TenureGroup,Complain,用户数,流失人数,流失率,平均订单数,样本提示
1,0‑12个月,1,1019,452,0.44,2.62,可观察
0,0‑12个月,0,2533,394,0.16,2.53,可观察
3,12‑24个月,1,439,56,0.13,3.27,可观察
2,12‑24个月,0,1135,46,0.04,3.79,可观察
4,24‑36个月,0,356,0,0.00,3.82,可观察
5,24‑36个月,1,144,0,0.00,3.40,可观察
6,36个月以上,0,2,0,0.00,2.50,小样本
7,36个月以上,1,2,0,0.00,1.50,小样本


In [30]:
# 检查点4
assert dim_1 in df.columns and dim_2 in df.columns, "两个维度必须是有效字段"
assert dim_1 != dim_2, "两个维度不能相同"
assert isinstance(cross_analysis, pd.DataFrame), "cross_analysis应为DataFrame"
assert {"用户数", "流失率", "样本提示"}.issubset(cross_analysis.columns), "双维表缺少必需列"
assert set(cross_analysis["样本提示"]).issubset({"小样本", "可观察"}), "样本提示取值不正确"
print("检查点4通过")

检查点4通过


### 双维分析记录

**最值得关注的组合：**  
最值得关注的组合：0‑12 个月且有投诉的用户分组。

**该组合的样本量与流失率：**  
样本充足标记为可观察，该分组流失率显著高于其余人群。

**为什么不能直接下因果结论：**  
目前仅能说明投诉和高流失存在相关关系，流失还会受满意度、下单频次等因素影响，不能判定投诉是流失的原因，还需要结合更多指标进一步验证

## 任务5：报表输出与回读验证（必做）

In [31]:
# TODO：将三个表导出到OUTPUT_DIR
outputs = {
    "overall_metrics.csv": overall_metrics,
    "segment_analysis.csv": segment_analysis,
    "cross_analysis.csv": cross_analysis,
}

# TODO：循环导出csv文件，参数 index=False, encoding="utf-8-sig"
for filename, table in outputs.items():
    save_path = OUTPUT_DIR / filename
    table.to_csv(save_path, index=False, encoding="utf-8-sig")

    reload_df = pd.read_csv(save_path)
    print(f"{filename} 导出后读取的shape：{reload_df.shape}")

overall_metrics.csv 导出后读取的shape：(10, 2)
segment_analysis.csv 导出后读取的shape：(4, 6)
cross_analysis.csv 导出后读取的shape：(8, 7)


In [32]:
# 检查点5
for filename, table in outputs.items():
    path = OUTPUT_DIR / filename
    assert path.exists(), f"缺少输出文件：{filename}"
    reloaded = pd.read_csv(path)
    assert reloaded.shape == table.shape, f"{filename}回读形状不一致"
print("检查点5通过：三个CSV均已成功导出并回读。")

检查点5通过：三个CSV均已成功导出并回读。


## 任务6：结论、限制与建议（必做）

### 结论1

在0‑12 个月新用户中，指标为流失率，与长期留存用户相比显著更高，对应证据表：segment_analysis

### 结论2

0‑12 个月且产生投诉的用户，相比同期无投诉用户流失率更高，对应证据表：cross_analysis

### 结论3

使用年限更长的用户，平均订单数和平均返现金额更高，用户粘性更强，对应证据表：segment_analysis

### 分析限制

缺少投诉问题详情、售后处理结果的数据，无法判定售后处理质量对流失的影响

### 运营建议与验证方式

给 0‑12 个月有投诉的新用户发放补偿优惠券；后续开展对照实验，获取实验组和对照组的后续留存数据，判断福利政策能否降低流失率

## 拓展任务（选做）

In [33]:
# 可选方向：
# 1. 使用qcut构建订单活跃度分层；
# 2. 设计供第6天绘图使用的长表；
# 3. 对反直觉结果提出两种数据核查方法。

# TODO（选做2）
# 选取需要的指标，宽表
wide_df = df[["CustomerID", "Churn", "Tenure", "OrderCount", "CashbackAmount", "HourSpendOnApp"]]
# melt函数转换成长格式，适配后续seaborn绘图
long_df = wide_df.melt(
    id_vars = ["CustomerID", "Churn"],
    value_vars = ["Tenure", "OrderCount", "CashbackAmount", "HourSpendOnApp"],
    var_name = "指标名称",
    value_name = "指标数值"
)
display(long_df)

,CustomerID,Churn,指标名称,指标数值
0,50001,1,Tenure,4.00
1,50002,1,Tenure,9.00
2,50003,1,Tenure,9.00
3,50004,1,Tenure,0.00
4,50005,1,Tenure,0.00
...,...,...,...,...
22515,55626,0,HourSpendOnApp,3.00
22516,55627,0,HourSpendOnApp,3.00
22517,55628,0,HourSpendOnApp,3.00
22518,55629,0,HourSpendOnApp,4.00


## 提交前检查

- [✅️] 已填写组号、成员和专题；
- [✅️] 已重启内核并从头运行成功；
- [✅️] 所有比例表都包含样本量；
- [✅️] 三个CSV已导出并回读；
- [✅️] 至少3条结论可对应到具体表格；
- [✅️] 已写明分析限制和验证建议；
- [✅️] 没有把返现写成消费额，没有把相关写成因果。